# Full-corpus paper embeddings (Kaggle / GPU only)

Generates `embeddings.npy` for all 119,919 rows of `papers.parquet` with
`all-MiniLM-L6-v2`, plus a small alignment check file. **Do not run this
locally** — it needs a GPU and the full cleaned dataset.

Before running:

1. Attach the `arxiv-papers-cleaned` Kaggle dataset to this notebook. With the
   current Kaggle layout its `papers.parquet` mounts at
   `/kaggle/input/datasets/mohammadibraheem71/arxiv-papers-cleaned/papers.parquet`;
   the run cell also falls back to the older
   `/kaggle/input/arxiv-papers-cleaned/papers.parquet` path.
2. Set **Settings -> Accelerator -> GPU**. If you change this after the kernel
   starts, restart the kernel.
3. Make sure the GitHub repo has been pushed, so the clone below contains
   `src/embed_all.py`.

This notebook only generates and verifies the files. Publishing the output
dataset is done manually afterwards.

In [ ]:
# Clone the repository and make it the working directory.
# The script inserts the repo root into sys.path itself, but doing it here too
# lets the GPU-check cell import src.config directly.
import os
import subprocess
import sys
from pathlib import Path

REPO = "research_paper_discovery"
repo_dir = Path("/kaggle/working") / REPO
if not (repo_dir / "src" / "embed_all.py").exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/MohammadIbraheem71/research_paper_discovery.git",
         str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("cwd:", os.getcwd())
print("script exists:", (repo_dir / "src" / "embed_all.py").exists())

In [ ]:
# Dependencies and GPU check. Stop here if no GPU is visible.
!pip -q install sentence-transformers pandas pyarrow

import torch
from src.config import EMBEDDING_DIM, EMBEDDING_MODEL

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Set Settings -> Accelerator -> GPU, then "
        "Restart & Run All before continuing."
    )
print("GPU:", torch.cuda.get_device_name(0))
print("model:", EMBEDDING_MODEL, "dim:", EMBEDDING_DIM)

In [ ]:
# Generate the embeddings. This is the long-running cell (~119,919 rows).
import subprocess
import sys
from pathlib import Path

# Kaggle changed the input mount layout; try the current path first and keep
# the older slug as a fallback.
PAPERS_CANDIDATES = [
    "/kaggle/input/datasets/mohammadibraheem71/arxiv-papers-cleaned/papers.parquet",
    "/kaggle/input/arxiv-papers-cleaned/papers.parquet",
]
PAPERS_PATH = next((p for p in PAPERS_CANDIDATES if Path(p).exists()), None)
assert PAPERS_PATH, f"papers.parquet not found; checked: {PAPERS_CANDIDATES}"

cmd = [
    sys.executable, "src/embed_all.py",
    "--papers", PAPERS_PATH,
    "--out", "/kaggle/working/embeddings.npy",
    "--check", "results/embeddings_check.json",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Verify row order, shape and normalization before doing anything else.
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

check = json.load(open("results/embeddings_check.json"))
print(json.dumps(check, indent=2))

ids = pd.read_parquet(PAPERS_PATH, columns=["id"])["id"]
digest = hashlib.sha256("\n".join(ids.astype(str).tolist()).encode("utf-8")).hexdigest()
assert digest == check["id_sha256"], "ID hash mismatch - wrong row order!"
assert len(ids) == check["rows"]

emb = np.load("/kaggle/working/embeddings.npy", mmap_mode="r")
assert emb.shape == (check["rows"], check["embedding_dim"]), emb.shape
sample = np.asarray(emb[::1000], dtype=np.float32)
assert np.allclose(np.linalg.norm(sample, axis=1), 1.0, atol=1e-5)
print(
    "VERIFIED:", emb.shape, "hash OK, norms OK, size",
    round(Path("/kaggle/working/embeddings.npy").stat().st_size / 1e6, 1), "MB",
)

## Manual handoff

This notebook does **not** publish anything.

1. Download `results/embeddings_check.json` from the **Output** tab and copy it
   into the repo.
2. Publish `embeddings.npy` as a public Kaggle Dataset yourself.
3. Update the embeddings dataset link under **Dataset** in `README.md` if you published a new one.
4. Commit `README.md` and `results/embeddings_check.json`, then push.

`embeddings.npy` is intentionally gitignored and must not be committed to Git.